# Разбор A. Один файл, два отпечатка: где искать расхождение?

Вы готовите Notebook по истрадефиллину для передачи коллеге. В папке лежит `compounds.json`, рядом — рассчитанные результаты. Коллега сможет открыть файл, но как ей установить, что использована именно ваша версия данных? Имя файла этого не гарантирует. И даже совпадение входного файла ещё не обещает совпадения всех численных результатов.

Здесь мы разберём небольшой диагностический случай: добавим пробел в **копию** JSON. Изменятся ли данные? Прежде чем запускать код, запишите два отдельных прогноза: изменится ли SHA-256 и изменится ли объект после чтения JSON. К концу разбора вы объясните, почему ответы могут различаться, и подготовите паспорт передачи расчёта. Химическую идентичность записей будем проверять в M03.

Я начинаю с решения, которое нам действительно нужно: разрешить коллеге продолжить исследование с переданного состояния. Для этого слово «тот же» слишком неопределённо. Один человек имеет в виду файл, другой — прочитанные записи, третий — химическую структуру. Если заранее не разделить эти значения, мы будем исправлять разные проблемы одинаковой командой «запустите ещё раз».

Поэтому я выстраиваю проверку от более простого к более содержательному. Сначала устанавливаю, сохранились ли байты. Затем проверяю, как их прочитал JSON-парсер. После этого спрашиваю, сохранились ли записи, значимые для дальнейшего алгоритма. Химическая форма и экспериментальный образец потребуют собственных свидетельств. Такая очередность даёт возможность остановиться там, где возникло первое расхождение, и объяснить его.

## 1. Сначала определим, с чем работает программа

SHA-256 — отпечаток последовательности байтов. В этом уроке он помогает обнаружить изменение файла. Он не подтверждает правильность источника, химическую форму или добросовестность поставщика данных.

Служебный блок ниже находит корень курса и создаёт рабочую папку. В `sys.path` добавляется каталог общих функций; это путь поиска модулей во время выполнения. Настройка Pylance для редактора задаётся отдельно в `.vscode/settings.json`.

In [2]:
from pathlib import Path
import sys
import json
import hashlib
import platform
from importlib.metadata import version
from IPython.display import display

# Ищем корень по реестру курса, а не по имени папки пользователя.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'programm/course.json').is_file())
sys.path.insert(0, str(ROOT / 'practicuum/shared'))
from course_support import DATA, workspace, write_json, init_dossier

OUT = workspace('M02-L01', 'case')


Можно было бы начать с печати содержимого или сравнения размеров. Я выбираю отпечаток, потому что одинаковый размер не исключает изменения значения: строка с одной цифрой может быть заменена другой строкой той же длины. Отпечаток помогает обнаружить различие, но не локализует его. Поэтому после него всё равно потребуется содержательная проверка.

Я также не начинаю с переустановки библиотек. Если файл уже отличается, сначала нужно понять это отличие. Изменение среды одновременно с изменением данных затруднит диагностику: после повторного запуска мы не будем знать, что именно повлияло на результат.

Прочитаем файл сначала как байты. Метод `read_bytes()` ещё ничего не знает о молекулах или формате JSON. Затем явно декодируем UTF-8 и разберём структуру документа. Так мы не смешиваем чтение файла с интерпретацией его содержимого.

In [3]:
path = DATA / 'compounds.json'
raw = path.read_bytes()
digest = hashlib.sha256(raw).hexdigest()
records = json.loads(raw.decode('utf-8'))

assert isinstance(records, list) and records
card = next(row for row in records if row['name'] == 'istradefylline')
display({'bytes': len(raw), 'sha256': digest,
         'record_count': len(records), 'selected_name': card['name']})


{'bytes': 4587,
 'sha256': 'dc2e00bec57f6ec4b3db654a0fb64e48272b091c0dd536e30521cb501ba6c267',
 'record_count': 7,
 'selected_name': 'istradefylline'}

`bytes` описывает размер файла, `record_count` — число прочитанных записей. Это разные проверки. Непустой файл может содержать не тот набор данных. Выбранное имя сейчас лишь находит запись: её формулу, граф и стереохимию мы этим не проверили.

## 2. Создадим расхождение, причину которого знаем

Теперь мне нужен опыт, у которого заранее понятна причина изменения. Если просто взять два чужих файла с разными отпечатками, различие можно объяснить десятками способов. В нашем опыте меняется только конечный пробел. Точная копия показывает, что механизм сравнения способен распознать неизменённый вход; копия с пробелом показывает, что байтовое сравнение замечает даже изменение, которое парсер затем игнорирует.

Почему я не удаляю пробелы во всём документе? Внутри строкового значения пробел может быть частью данных. Наше вмешательство ограничено концом корректного JSON; вывод о нём нельзя автоматически распространить на другие места файла.

Сделаем две рабочие копии: точную и с пробелом после конца документа. Исходник в `shared/data` оставим неизменным. Последовательность `b' '` означает один байт пробела, а не новое поле в JSON.

In [4]:
exact_copy = OUT / 'compounds_exact.json'
spaced_copy = OUT / 'compounds_with_space.json'
exact_copy.write_bytes(raw)
spaced_copy.write_bytes(raw + b' ')

comparisons = []
for candidate in [exact_copy, spaced_copy]:
    payload = candidate.read_bytes()
    comparisons.append({
        'file': candidate.name,
        'bytes': len(payload),
        'same_sha256': hashlib.sha256(payload).hexdigest() == digest,
        'same_parsed_object': json.loads(payload.decode('utf-8')) == records,
    })
display(comparisons)


[{'file': 'compounds_exact.json',
  'bytes': 4587,
  'same_sha256': True,
  'same_parsed_object': True},
 {'file': 'compounds_with_space.json',
  'bytes': 4588,
  'same_sha256': False,
  'same_parsed_object': True}]

Вернитесь к прогнозу. У точной копии оба сравнения должны дать `True`. У копии с пробелом отпечаток отличается, но прочитанный объект равен исходному: конечный пробел не добавляет запись или поле.

Равенство здесь относится к полученным Python-объектам. Это не универсальный критерий химической эквивалентности и не способ игнорировать любое изменение файла. Например, перестановка записей меняет порядок списка; значение этого порядка зависит от последующего алгоритма.

In [5]:
same_object = comparisons[1]['same_parsed_object']
same_bytes = raw == spaced_copy.read_bytes()
assert comparisons[0]['same_sha256']
assert same_object and not same_bytes
assert path.read_bytes() == raw  # Контрольный опыт не изменил источник.


Мы обнаружили байтовое расхождение и объяснили его причину. Из этого нельзя сделать вывод, что все отличающиеся файлы безопасны. При неизвестном изменении сначала выясняют, какие записи и поля затронуты, затем решают, нужно ли повторять последующие расчёты.

Полученное сочетание «отпечатки различаются, прочитанные объекты равны» меняет решение. Мне больше не нужно искать исчезнувшую молекулу в этом конкретном опыте: известное вмешательство не изменило объект после разбора. Но исходный файл всё равно остаётся отдельной версией байтов; если протокол требует точной передачи, я передам исходную копию, а не объявлю разные версии одной и той же.

Если бы объекты оказались различны, я пошёл бы к первому изменённому полю и спросил, участвует ли оно в расчёте. Изменение подписи и изменение молярной массы потребуют разных решений. Затем проверил бы, по каким ключам соединяются таблицы: перестановка строк может быть безвредной при соединении по устойчивому ID и опасной при сопоставлении по позиции.

## 3. Что передать вместе с данными

Файл и его отпечаток отвечают на вопрос «какие байты?». Для вопроса «какой программой обработали?» нужны интерпретатор, версии библиотек, код и параметры. Сохраним текущую среду и отпечаток файла закреплённых зависимостей. `uv.lock` описывает согласованную установку, а зафиксированные версии показывают фактически доступные пакеты; одно не подменяет другое.

In [6]:
environment = {
    'python': sys.version,
    'executable': sys.executable,
    'platform': platform.platform(),
    'packages': {name: version(name) for name in ['nbformat', 'matplotlib']},
    'uv_lock_sha256': hashlib.sha256(
        (ROOT / 'scripts/environment/uv.lock').read_bytes()).hexdigest(),
}
input_manifest = {
    'path': 'practicuum/shared/data/compounds.json',
    'bytes': len(raw), 'sha256': digest, 'encoding': 'utf-8',
}
write_json(OUT / 'environment.json', environment)
write_json(OUT / 'input_manifest.json', input_manifest)
display({'interpreter': environment['executable'],
         'packages': environment['packages']})


{'interpreter': 'C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\scripts\\environment\\.venv-base\\Scripts\\python.exe',
 'packages': {'nbformat': '5.11.1', 'matplotlib': '3.11.2'}}

Абсолютный путь к интерпретатору на другой машине будет иным — это ожидаемо. Отпечаток исходного файла должен совпасть при передаче точной копии. Для будущих численных результатов потребуется отдельный обоснованный допуск: одинаковый вход сам по себе не устраняет различия алгоритмов, библиотек и арифметики.

Теперь мысленно повторим передачу на другой машине. Другой путь к Python ожидаем: путь зависит от установки. Другая версия библиотеки уже требует выяснить, какие операции изменились. Иное число при совпавших входах заставляет проверить параметры алгоритма, случайность и численную арифметику. Допуск для числа выбирают из смысла метода; совпадение двух хешей не выдаёт такой допуск автоматически.

Поэтому паспорт нужен как маршрут диагностики. Коллега должна не просто увидеть перечень версий, а понять, что сравнивать первым, где искать причину и какие результаты пока можно использовать. Это и есть практический результат урока.

## 4. Запишем вывод, который действительно получили

Результат этого опыта — различение байтовой и структурной идентичности. Мы не измеряли свойства вещества. В досье попадёт вычисленный результат с происхождением входа, а не утверждение о проверенной химической форме.

In [7]:
result = {
    'kind': 'calculated', 'input_sha256': digest,
    'semantic_identity': same_object,
    'same_bytes_after_whitespace_change': same_bytes,
    'comparisons': comparisons, 'packages': environment['packages'],
    'chemical_identity': 'NOT_ESTABLISHED_BY_FILE_HASH',
}
write_json(OUT / 'result.json', result)
from dossier_records import append_record
receipt = append_record(
    init_dossier(), 'M02-L01', 'A', result, ROOT,
    inputs=['practicuum/shared/data/compounds.json'],
    method='SHA-256 of bytes and equality of parsed JSON objects',
    decision='Explain file differences before comparing downstream calculations',
)
print('Сохранена версия записи:', receipt)


Сохранена версия записи: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\reasoning-full-20261120\workspace\my_research\evidence\M02-L01\20261120T105309214287Z-A-55867b57dd864033ac6e7e9fa1b33f5c.json


Выберите действие для трёх ситуаций: отличается только путь к Python; отличается отпечаток входа; совпадает вход, но отличаются результаты расчёта. Для каждой назовите следующую проверку. Если везде ответили «переустановить Python», вернитесь к тому, что именно проверяет каждое сравнение.

Теперь коллега получает два паспорта и объяснение контролируемого расхождения. В лабораторной вы подготовите передачу данных по аторвастатину, а в следующем уроке научитесь читать таблицу так, чтобы типы и единицы не терялись при обработке.